# Scenario Impact playground: ENSO / HeatDry / HeatDryYoY counterfactuals

Standalone, one-country-at-a-time sandbox for the same "no-ENSO" / "no-HeatDry(YoY)"
counterfactual comparison shown in the dashboard's Scenario Impacts tab -- but without
touching `Dash_Input/gvar_forecast_results.pkl` or any dashboard code. Nothing here
writes any file; rerun any cell after changing a config value.

Reuses only the shared, tested Kalman/EM math in `trp/kalman_core.py`
(`init_from_varx_rolling`, `kf_e_step_store`, `rts_smoother`, `em_m_step_update`,
`run_kf_em`) -- same as `structural_break/GVAR_LLM_pickle.py` and
`analysis/validation/kf_em_diagnostics.py`. The forecast rollout itself (a simple
recursive `y_hat = H_t @ theta`) is reimplemented inline below, deliberately kept
tiny and dependency-free so it's easy to read and modify.

Things you can freely change and rerun from the CONFIG cell onward:
- `COUNTRY` (must be one of the 12 core countries)
- `LAGS`
- `USE_ZSCORE` (standardize Y/Z before fitting, or fit on raw units)
- `EXO_BASE` (baseline economic drivers)
- `HEAT_VAR` (`"HeatDry"`, `"HeatDryYoY"`, or `None` for ENSO-only)
- `WINDOW`, `MAX_EM_ITER`, `UPDATE_P0`
- `ENSO_SCENARIO` (edit the future ENSO path directly)
- `FORECAST_HORIZON_Q`, `MC_SIMULATIONS`

In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

_ROOT = Path.cwd()
while not (_ROOT / "trp" / "kalman_core.py").is_file() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
assert (_ROOT / "trp" / "kalman_core.py").is_file(), "Run this notebook from inside the TRP_Dashboard repo."
for p in (_ROOT, _ROOT / "structural_break"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from trp.kalman_core import run_kf_em  # noqa: E402
from trp.inputs import load_gvar_panel  # noqa: E402

print("Repo root:", _ROOT)

Repo root: /Users/poppy/Desktop/TRP_Dashboard


## Config -- edit these and rerun everything below

In [10]:
CORE_COUNTRIES = ["BRA", "CHL", "COL", "MEX", "KEN", "ZAF", "IND", "IDN", "THA", "PER", "PHL", "EGY"]

# ---- pick one country at a time ----
COUNTRY = "THA"
assert COUNTRY in CORE_COUNTRIES, f"{COUNTRY!r} is not one of the 12 core countries: {CORE_COUNTRIES}"

# ---- model spec (freely adjustable) ----
ENDO = ["GDP_YoY", "CPI_YoY", "FX_YoY", "EX_YoY"]
EXO_BASE = ["COMMODITY_YoY", "US_GDP_YoY"]   # always-included baseline economic drivers
HEAT_VAR = "HeatDry"                          # "HeatDry", "HeatDryYoY", or None for ENSO-only
LAGS = 1
USE_ZSCORE = True                             # False = fit directly on raw (unstandardized) units
WINDOW = 40                                   # rolling-VARX burn-in window (init_from_varx_rolling)
MAX_EM_ITER = 30
TOL = 1e-4
UPDATE_P0 = True                              # matches current production default; try False too

EXO_USE = EXO_BASE + ["ENSO"] + ([HEAT_VAR] if HEAT_VAR else [])
print("ENDO:", ENDO)
print("EXO_USE:", EXO_USE)

# ---- forecast scenario ----
FORECAST_HORIZON_Q = 8       # quarters to forecast past the last complete historical quarter
MC_SIMULATIONS = 200         # lower than the dashboard's 500 for faster iteration; 0 = no band
MC_SEED = 20260531

# Edit this directly to try a different ENSO path (same numbers as the dashboard's
# FORECAST_ENSO_MEAN by default). Any target quarter not listed here falls back to
# the last value in the dict.
ENSO_SCENARIO = {
    "2025Q3": -0.6055,
    "2025Q4": -0.9034,
    "2026Q1": -0.6499,
    "2026Q2": 0.4286,
    "2026Q3": 1.7784,
    "2026Q4": 2.2875,
    "2027Q1": 1.8143,
}

ENDO: ['GDP_YoY', 'CPI_YoY', 'FX_YoY', 'EX_YoY']
EXO_USE: ['COMMODITY_YoY', 'US_GDP_YoY', 'ENSO', 'HeatDry']


## Load + prep this country's data

Real panel values are always preferred where available; `ENSO_SCENARIO` (or the
country's own trailing 10-year mean for `HEAT_VAR`) only fills in quarters the
panel doesn't cover yet.

In [11]:
def prep_country_data(country, endo, exo_use, lags, use_zscore, window):
    df = load_gvar_panel()
    g = df[df["country"] == country].sort_values("quarter").reset_index(drop=True)
    mask = np.isfinite(g[endo].to_numpy(float)).all(axis=1) & np.isfinite(g[exo_use].to_numpy(float)).all(axis=1)
    g = g.loc[mask].reset_index(drop=True)
    assert len(g) >= lags + 5, f"Only {len(g)} usable rows for {country} with EXO_USE={exo_use}"

    Yd_raw = g[endo].to_numpy(float)
    Xd_raw = g[exo_use].to_numpy(float)
    if use_zscore:
        mu_y, sd_y = Yd_raw.mean(axis=0), Yd_raw.std(axis=0) + 1e-8
        mu_x, sd_x = Xd_raw.mean(axis=0), Xd_raw.std(axis=0) + 1e-8
        Yd = (Yd_raw - mu_y) / sd_y
        Xd = (Xd_raw - mu_x) / sd_x
    else:
        mu_y, sd_y = np.zeros(len(endo)), np.ones(len(endo))
        mu_x, sd_x = np.zeros(len(exo_use)), np.ones(len(exo_use))
        Yd, Xd = Yd_raw, Xd_raw

    return {
        "g": g, "quarters": pd.to_datetime(g["quarter"]),
        "Yd": Yd, "Xd": Xd, "Yd_raw": Yd_raw, "Xd_raw": Xd_raw,
        "mu_y": mu_y, "sd_y": sd_y, "mu_x": mu_x, "sd_x": sd_x,
    }


prep = prep_country_data(COUNTRY, ENDO, EXO_USE, LAGS, USE_ZSCORE, WINDOW)
print(f"{COUNTRY}: {len(prep['g'])} usable quarters, {prep['quarters'].min().date()} .. {prep['quarters'].max().date()}")

THA: 60 usable quarters, 2011-01-01 .. 2025-10-01


## Fit the EM/Kalman model (reuses trp.kalman_core.run_kf_em, no pickle involved)

In [12]:
res = run_kf_em(
    Y=prep["Yd"], Z=prep["Xd"], lags=LAGS, window=WINDOW,
    max_em_iter=MAX_EM_ITER, tol=TOL, em_damping=0.0,
    verbose=True, update_P0=UPDATE_P0,
)
hist = res["em_history"]
print(f"\nconverged={len(hist['obj']) < MAX_EM_ITER}  n_iter={len(hist['obj'])}  "
      f"final_ll={hist['log_likelihood'][-1]:.2f}  final_obj={hist['obj'][-1]:.5f}")

[EM] iter=01, obj=1.190030, ll=-219.866, trQ=4.249874e-02, trR=1.307696e+00
[EM] iter=02, obj=1.272997, ll=-182.029, trQ=4.290497e-02, trR=1.448018e+00
[EM] iter=03, obj=1.276364, ll=-177.724, trQ=4.320743e-02, trR=1.460951e+00
[EM] iter=04, obj=1.267647, ll=-175.582, trQ=4.359946e-02, trR=1.452676e+00
[EM] iter=05, obj=1.256430, ll=-173.843, trQ=4.414190e-02, trR=1.440022e+00
[EM] iter=06, obj=1.244701, ll=-172.234, trQ=4.486123e-02, trR=1.426505e+00
[EM] iter=07, obj=1.233082, ll=-170.699, trQ=4.576649e-02, trR=1.413132e+00
[EM] iter=08, obj=1.221838, ll=-169.235, trQ=4.685340e-02, trR=1.400272e+00
[EM] iter=09, obj=1.211078, ll=-167.852, trQ=4.810687e-02, trR=1.388060e+00
[EM] iter=10, obj=1.200823, ll=-166.559, trQ=4.950407e-02, trR=1.376510e+00
[EM] iter=11, obj=1.191038, ll=-165.358, trQ=5.101825e-02, trR=1.365573e+00
[EM] iter=12, obj=1.181667, ll=-164.246, trQ=5.262219e-02, trR=1.355170e+00
[EM] iter=13, obj=1.172647, ll=-163.214, trQ=5.429089e-02, trR=1.345217e+00
[EM] iter=14

## Build the forecast: full scenario, no-ENSO, and no-HeatDry(YoY)

Same three ingredients the dashboard's `forecast_country_from_em` uses: the last
filtered state, the EM-converged Q/R, and a simple deterministic rollout with an
optional Monte-Carlo band driven by Q (mirrors `_roll_kf_forecast` in
`analysis/Dash_Output/gvar_kf_forecast.py`, reimplemented here so this notebook has
no import dependency on the dashboard/pickle-generation code).

In [13]:
def quarter_start(ts):
    return pd.Timestamp(ts).to_period("Q").to_timestamp()


def period_key(ts):
    p = pd.Timestamp(ts).to_period("Q")
    return f"{p.year}Q{p.quarter}"


pack = res["e_step_store"]
valid_idx = np.where(pack["valid_mask"])[0]
last_t = valid_idx[-1]
mY, mX = len(ENDO), len(EXO_USE)
m = LAGS * mY + mX

theta_last = pack["theta_filt"][last_t].reshape(-1, 1)
P_last = pack["P_filt"][last_t]
Q, R = res["Q"], res["R"]

last_hist_q = quarter_start(prep["quarters"].iloc[last_t])
target_quarters = [ (last_hist_q.to_period("Q") + h).to_timestamp() for h in range(1, FORECAST_HORIZON_Q + 1) ]
print("Last complete historical quarter:", last_hist_q.date())
print("Forecasting:", [period_key(q) for q in target_quarters])

Last complete historical quarter: 2025-10-01
Forecasting: ['2026Q1', '2026Q2', '2026Q3', '2026Q4', '2027Q1', '2027Q2', '2027Q3', '2027Q4']


In [14]:
def build_exo_forecast(target_quarters, exo_use, country_raw_df, enso_scenario, heat_var):
    """Real panel value preferred; ENSO_SCENARIO / trailing mean only fill gaps."""
    g = country_raw_df.set_index(country_raw_df["quarter"].apply(quarter_start))
    heat_mean = np.nan
    if heat_var is not None and heat_var in g.columns:
        last_q = g.index.max()
        window = g[g.index > last_q - pd.DateOffset(years=10)]
        heat_mean = pd.to_numeric(window[heat_var], errors="coerce").mean()

    rows, sources = [], []
    for tq in target_quarters:
        row, src = {}, {}
        for var in exo_use:
            actual = g[var].loc[tq] if (var in g.columns and tq in g.index) else np.nan
            actual = pd.to_numeric(pd.Series([actual]), errors="coerce").iloc[0]
            if np.isfinite(actual):
                row[var], src[var] = float(actual), "panel"
            elif var == "ENSO":
                row[var] = enso_scenario.get(period_key(tq), list(enso_scenario.values())[-1])
                src[var] = "forecast"
            elif var == heat_var:
                row[var] = heat_mean
                src[var] = "forecast_10yr_mean"
            else:
                row[var] = np.nan
                src[var] = "missing"
        rows.append(row)
        sources.append(src)
    return pd.DataFrame(rows, index=target_quarters), pd.DataFrame(sources, index=target_quarters)


exo_fc, exo_src = build_exo_forecast(target_quarters, EXO_USE, prep["g"], ENSO_SCENARIO, HEAT_VAR)
print(exo_fc)
print()
print(exo_src)

            COMMODITY_YoY  US_GDP_YoY    ENSO  HeatDry
2026-01-01            NaN         NaN -0.6499  0.44185
2026-04-01            NaN         NaN  0.4286  0.44185
2026-07-01            NaN         NaN  1.7784  0.44185
2026-10-01            NaN         NaN  2.2875  0.44185
2027-01-01            NaN         NaN  1.8143  0.44185
2027-04-01            NaN         NaN  1.8143  0.44185
2027-07-01            NaN         NaN  1.8143  0.44185
2027-10-01            NaN         NaN  1.8143  0.44185

           COMMODITY_YoY US_GDP_YoY      ENSO             HeatDry
2026-01-01       missing    missing  forecast  forecast_10yr_mean
2026-04-01       missing    missing  forecast  forecast_10yr_mean
2026-07-01       missing    missing  forecast  forecast_10yr_mean
2026-10-01       missing    missing  forecast  forecast_10yr_mean
2027-01-01       missing    missing  forecast  forecast_10yr_mean
2027-04-01       missing    missing  forecast  forecast_10yr_mean
2027-07-01       missing    missing  forec

In [15]:
def build_H(y_buf, z_row, mY, m, lags):
    x_t = np.concatenate(y_buf + [z_row])
    H = np.zeros((mY, m * mY))
    for j in range(mY):
        H[j, j * m:(j + 1) * m] = x_t
    return H


def roll_forecast_mean(theta, y_buf0, z_fc, mY, m, lags):
    y_buf = [y.copy() for y in y_buf0]
    y_hat = np.zeros((len(z_fc), mY))
    for h in range(len(z_fc)):
        H = build_H(y_buf, z_fc[h], mY, m, lags)
        yhat = (H @ theta).ravel()
        y_hat[h] = yhat
        y_buf = [yhat.copy()] + y_buf[:-1]
    return y_hat


def roll_forecast_band(theta, Q, y_buf0, z_fc, mY, m, lags, n_sims, seed):
    if n_sims <= 0:
        return None, None
    L = np.linalg.cholesky(0.5 * (Q + Q.T) + 1e-8 * np.eye(Q.shape[0]))
    rng = np.random.default_rng(seed)
    paths = np.full((n_sims, len(z_fc), mY), np.nan)
    for s in range(n_sims):
        beta = theta.copy()
        y_buf = [y.copy() for y in y_buf0]
        for h in range(len(z_fc)):
            beta = beta + L @ rng.standard_normal((theta.shape[0], 1))
            H = build_H(y_buf, z_fc[h], mY, m, lags)
            yhat = (H @ beta).ravel()
            paths[s, h] = yhat
            y_buf = [yhat.copy()] + y_buf[:-1]
    sd = np.nanstd(paths, axis=0, ddof=1)
    return sd, paths


def zscore_exo(exo_df, mu_x, sd_x, exo_use):
    z = np.zeros((len(exo_df), len(exo_use)))
    for j, col in enumerate(exo_use):
        z[:, j] = (exo_df[col].to_numpy(float) - mu_x[j]) / sd_x[j]
    return np.where(np.isfinite(z), z, 0.0)


z_fc = zscore_exo(exo_fc, prep["mu_x"], prep["sd_x"], EXO_USE)
y_buf0 = [prep["Yd"][last_t - i, :].copy() for i in range(1, LAGS + 1)]

y_hat = roll_forecast_mean(theta_last, y_buf0, z_fc, mY, m, LAGS)
sd_band, _ = roll_forecast_band(theta_last, Q, y_buf0, z_fc, mY, m, LAGS, MC_SIMULATIONS, MC_SEED)

# --- no-ENSO counterfactual: zero only the *forecast-sourced* ENSO rows ---
enso_j = EXO_USE.index("ENSO")
z_fc_enso0 = z_fc.copy()
enso0_mask = (exo_src["ENSO"] != "panel").to_numpy()
z_fc_enso0[enso0_mask, enso_j] = (0.0 - prep["mu_x"][enso_j]) / prep["sd_x"][enso_j]
y_hat_enso0 = roll_forecast_mean(theta_last, y_buf0, z_fc_enso0, mY, m, LAGS)

# --- no-HeatDry(YoY) counterfactual, only if a heat/dryness driver is active ---
y_hat_heat0 = None
if HEAT_VAR is not None:
    heat_j = EXO_USE.index(HEAT_VAR)
    z_fc_heat0 = z_fc.copy()
    heat0_mask = (exo_src[HEAT_VAR] != "panel").to_numpy()
    z_fc_heat0[heat0_mask, heat_j] = (0.0 - prep["mu_x"][heat_j]) / prep["sd_x"][heat_j]
    y_hat_heat0 = roll_forecast_mean(theta_last, y_buf0, z_fc_heat0, mY, m, LAGS)

print("Forecast (z-scored, if USE_ZSCORE) shape:", y_hat.shape)

Forecast (z-scored, if USE_ZSCORE) shape: (8, 4)


## Plot -- pick which ENDO variable to show

In [16]:
RESPONSE_VAR = "GDP_YoY"
idx = ENDO.index(RESPONSE_VAR)


def to_raw(arr, j):
    """Undo z-scoring for plotting (no-op if USE_ZSCORE=False, since mu=0/sd=1 then)."""
    return arr[:, j] * prep["sd_y"][j] + prep["mu_y"][j]


def plot_counterfactual(y_main, y_cf, cf_name, title, hist_start=pd.Timestamp("2018-01-01")):
    hist_q = prep["quarters"]
    hist_y = prep["Yd_raw"][:, idx] if USE_ZSCORE else prep["Yd"][:, idx]
    hist_mask = hist_q >= hist_start
    fc_q = pd.DatetimeIndex(target_quarters)
    fc_main = to_raw(y_main, idx)
    fc_cf = to_raw(y_cf, idx) if y_cf is not None else None

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist_q[hist_mask], y=hist_y[hist_mask], mode="lines",
                              name="Actual history", line=dict(color="#1f77b4", width=2)))
    if sd_band is not None:
        band_raw = sd_band[:, idx] * prep["sd_y"][idx]
        fig.add_trace(go.Scatter(
            x=list(fc_q) + list(fc_q[::-1]),
            y=list(fc_main + band_raw) + list((fc_main - band_raw)[::-1]),
            fill="toself", fillcolor="rgba(31,119,180,0.12)",
            line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False,
        ))
    fig.add_trace(go.Scatter(x=fc_q, y=fc_main, mode="lines+markers", name="Forecasted scenario",
                              line=dict(color="#1f77b4", width=2)))
    if fc_cf is not None:
        fig.add_trace(go.Scatter(x=fc_q, y=fc_cf, mode="lines+markers", name=cf_name,
                                  line=dict(color="#111111", width=2, dash="dash")))
    fig.update_layout(title=title, xaxis_title="Quarter", yaxis_title=RESPONSE_VAR,
                       height=480, legend=dict(orientation="h", y=-0.2))
    fig.show()


plot_counterfactual(
    y_hat, y_hat_enso0, "No-ENSO counterfactual (ENSO = 0)",
    f"{COUNTRY}: {RESPONSE_VAR} -- forecasted ENSO vs no-ENSO counterfactual",
)

if HEAT_VAR is not None:
    plot_counterfactual(
        y_hat, y_hat_heat0, f"No-{HEAT_VAR} counterfactual",
        f"{COUNTRY}: {RESPONSE_VAR} -- forecasted scenario vs no-{HEAT_VAR} counterfactual",
    )

## Notes

- Re-running from the CONFIG cell picks up any change to `COUNTRY`, `LAGS`, `USE_ZSCORE`,
  `EXO_BASE`, `HEAT_VAR`, `WINDOW`, `MAX_EM_ITER`, `UPDATE_P0`, `ENSO_SCENARIO`,
  `FORECAST_HORIZON_Q`, or `MC_SIMULATIONS` -- nothing is cached to disk.
- Switch `RESPONSE_VAR` to `"CPI_YoY"`, `"FX_YoY"`, or `"EX_YoY"` and rerun just the
  plotting cell to see the other outcome variables without refitting the model.
- Setting `USE_ZSCORE = False` fits directly on raw-scale data; the EM's Q/R priors
  from `init_from_varx_rolling` were tuned assuming roughly standardized inputs, so
  convergence behavior may look different (worth comparing against `True`).
- This mirrors the dashboard's math but is **not** wired to
  `Dash_Input/gvar_forecast_results.pkl` -- to see a change reflected in the actual
  Streamlit app, you still need to rerun
  `python analysis/regenerate_dashboard_artifacts.py --skip-pipeline`.